# LoRA Layer-wise Factuality — Figures & Tables

Interactive, cell-by-cell rebuild of `src/lora_lens/visualize.py`. Each figure
or table gets its own cell so you can re-run, tweak, or skip any single output
without regenerating the rest. Plots are displayed inline **and** saved as PDF
under `<output_dir>/figures/...`, matching the layout the pipeline's
`visualize` stage already produces. Tables are rendered as `pandas`
DataFrames instead of LaTeX snippets.

Set `OUTPUT_DIR` and `LENS` in the configuration cell below; everything else
recomputes from those.

**Terminology.** The trajectory data calls the layer where the answer becomes
top-1 *and stays* top-1 through the output `settle_layer`. It is displayed
throughout as the **consistent layer**; the underlying column name is
unchanged so the pipeline's own outputs still line up.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, MaxNLocator, AutoMinorLocator
from scipy import stats as sstats
from IPython.display import display, Image
import io

from lora_lens import visualize as viz
from lora_lens.trajectory import moderator_data, per_prompt_trajectory, CLASS_ORDER

COND_ORDER = viz.COND_ORDER
CONDITION_LABELS = viz.CONDITION_LABELS

# ── Publication style ────────────────────────────────────────────────────
# Importing visualize.py applies its own rcParams (bold titles, 9pt text,
# 1.8pt lines) — a slide look. Everything below replaces it with the
# conventions of ACL/EMNLP/NeurIPS/ICML camera-ready figures:
#
#   * type sized for a figure printed at ~3.3in wide, not projected: 7pt
#     body, 6pt ticks, 6pt legends. Smaller than feels right on screen,
#     correct on the page.
#   * hairline rules (0.5pt) and pale gridlines, so data is the darkest ink
#     on the figure.
#   * no bold anywhere, no frames, no heavy markers.
#   * Type 42 (TrueType) font embedding — ACL and IEEE reject Type 3, which
#     is matplotlib's PDF default, so this matters at submission time.
FONT_STACK = ["DejaVu Sans", "Arial", "Helvetica", "Liberation Sans"]

plt.rcParams.update({
    # Type
    "font.family":         "sans-serif",
    "font.sans-serif":     FONT_STACK,
    "mathtext.fontset":    "dejavusans",
    "font.size":           7,
    "axes.titlesize":      7,
    "axes.titleweight":    "normal",
    "axes.labelsize":      7,
    "axes.labelpad":       2.0,
    "axes.titlepad":       3.5,
    "figure.titlesize":    7.5,
    "figure.titleweight":  "normal",
    # Rules and spines
    "axes.linewidth":      0.5,
    "axes.edgecolor":      "#4D4D4D",
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.axisbelow":      True,
    # Ticks
    "xtick.labelsize":     6,
    "ytick.labelsize":     6,
    "xtick.color":         "#4D4D4D",
    "ytick.color":         "#4D4D4D",
    "xtick.labelcolor":    "black",
    "ytick.labelcolor":    "black",
    "xtick.major.width":   0.5,
    "ytick.major.width":   0.5,
    "xtick.major.size":    2.0,
    "ytick.major.size":    2.0,
    "xtick.major.pad":     1.8,
    "ytick.major.pad":     1.8,
    "xtick.minor.width":   0.4,
    "ytick.minor.width":   0.4,
    "xtick.minor.size":    1.1,
    "ytick.minor.size":    1.1,
    # Data ink
    "lines.linewidth":     0.9,
    "lines.markersize":    2.4,
    "lines.markeredgewidth": 0.0,
    "patch.linewidth":     0.5,
    "hatch.linewidth":     0.4,
    # Grid
    "axes.grid":           True,
    "grid.linewidth":      0.3,
    "grid.alpha":          1.0,
    "grid.color":          "#E4E4E4",
    # Legend
    "legend.fontsize":         6,
    "legend.frameon":          False,
    "legend.handlelength":     1.4,
    "legend.handletextpad":    0.45,
    "legend.labelspacing":     0.22,
    "legend.columnspacing":    0.9,
    "legend.borderpad":        0.25,
    "legend.borderaxespad":    0.3,
    # Output
    "figure.dpi":          200,
    "savefig.dpi":         600,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.015,
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
})

# ── Palette ──────────────────────────────────────────────────────────────
# Okabe-Ito, the standard colour-blind-safe qualitative set. One mapping for
# the whole project, so a condition is the same colour in every figure.
OKABE_ITO = {
    "orange":      "#E69F00",
    "sky":         "#56B4E9",
    "green":       "#009E73",
    "yellow":      "#F0E442",
    "blue":        "#0072B2",
    "vermillion":  "#D55E00",
    "purple":      "#CC79A7",
    "grey":        "#999999",
}
C_BASE    = OKABE_ITO["grey"]
C_LORA    = OKABE_ITO["blue"]
C_ACCENT  = OKABE_ITO["vermillion"]
C_NEUTRAL = "#4D4D4D"

COLORS = {
    "known":     OKABE_ITO["blue"],
    "latent":    OKABE_ITO["sky"],
    "unknown":   OKABE_ITO["orange"],
    "synthetic": OKABE_ITO["vermillion"],
    "existing":  OKABE_ITO["blue"],
    "base":      C_BASE,
    "final":     C_LORA,
}
# Redundant encoding: the figures stay readable in greyscale and for readers
# who cannot separate the hues.
LINE_STYLES = {"known": "solid", "latent": (0, (4, 1.2)), "unknown": (0, (2, 1)),
               "synthetic": (0, (1, 1)), "existing": "solid"}
MARKERS     = {"known": "o", "latent": "D", "unknown": "s",
               "synthetic": "^", "existing": "o"}

# ── Figure sizes ─────────────────────────────────────────────────────────
# ACL/EMNLP \columnwidth is ~3.17in and \textwidth ~6.3in; NeurIPS/ICML
# text width is ~5.5in. These sizes drop into either without rescaling —
# which is what keeps the type at its intended point size on the page.
FIG_1COL     = (3.3, 1.95)
FIG_1COL_SQ  = (3.3, 2.5)
FIG_2COL     = (6.6, 2.1)
FIG_2COL_TALL = (6.6, 2.5)

# Display names: `settle_layer` in the data is the "consistent layer" in text.
LAYER_METRICS = [("first_layer", "First layer"),
                 ("settle_layer", "Consistent layer")]


In [ ]:
# ── Run selection ────────────────────────────────────────────────────────
# Points at an output_dir laid out the way the pipeline's `visualize` stage
# expects: results under <output_dir>/results, PDFs to <output_dir>/figures,
# conditions.parquet at the top of <output_dir>.
OUTPUT_DIR  = Path("../results/nlp_outputs")
RESULTS_DIR = OUTPUT_DIR / "results"
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Single knob for which lens every lens-dependent figure below uses.
LENS = "logit"

# In-axes titles help when reading the notebook, but a paper puts that text
# in the LaTeX caption instead and leaves the figure itself clean. Set False to
# re-export every PDF caption-ready, with no title or suptitle.
SHOW_TITLES = True

print(f"Results:  {RESULTS_DIR.resolve()}")
print(f"Figures:  {FIGURES_DIR.resolve()}")
print(f"Lens:     {LENS}")
print(f"Titles:   {'on' if SHOW_TITLES else 'off (caption-ready)'}")


In [ ]:
def show_and_save(fig, stem, *, lens=None, prefix="new_version"):
    """Display fig inline, save it as PDF under FIGURES_DIR, then close.

    Renders the PNG preview through an explicit buffer rather than relying on
    IPython's Figure repr formatter: that formatter is only registered by the
    `inline` matplotlib backend, and visualize.py's `matplotlib.use("Agg")`
    would otherwise leave plain `display(fig)` falling back to a text repr.
    """
    out_dir = FIGURES_DIR / lens if lens else FIGURES_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{prefix}_{stem}.pdf"
    fig.savefig(out)

    # Preview at 2x the print DPI so the small type stays legible on screen.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=400)
    plt.close(fig)
    display(Image(data=buf.getvalue(), width=int(fig.get_figwidth() * 108)))
    print(f"[notebook] saved {out}")


def title(ax, text, **kw):
    """Axes title, suppressed when exporting caption-ready figures."""
    if SHOW_TITLES:
        ax.set_title(text, **kw)


def suptitle(fig, text, **kw):
    """Figure title, suppressed when exporting caption-ready figures."""
    if SHOW_TITLES:
        fig.suptitle(text, **kw)


def layer_axis(ax, n_layers, *, step=4):
    """Layer x-axis: labelled ticks every `step`, unlabelled minor ticks every
    layer. Labelling all 25 layers is unreadable at 6pt, but the minor ticks
    still let a reader count off an individual layer."""
    ax.set_xlim(-0.5, n_layers + 0.5)
    ax.xaxis.set_major_locator(MultipleLocator(step))
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    ax.set_xlabel("Transformer layer")


def int_axis(ax, axis="x", *, step=2):
    """Integer axis with labels every `step` and minor ticks every unit."""
    target = ax.xaxis if axis == "x" else ax.yaxis
    target.set_major_locator(MultipleLocator(step))
    target.set_minor_locator(MultipleLocator(1))


def value_ticks(ax, values, axis="x"):
    """Ticks on the values actually sampled (ranks, checkpoint steps), where a
    fixed interval would produce mostly empty gradations."""
    vals = sorted(set(values))
    (ax.set_xticks if axis == "x" else ax.set_yticks)(vals)


def wilson_ci(k, n, z=1.96):
    """Wilson score interval for a binomial proportion, as arrays in [0, 1].

    Preferred over the normal approximation here because several curves run
    against 0% or 100%, where the naive interval spills outside [0, 1].
    """
    k, n = np.asarray(k, float), np.asarray(n, float)
    with np.errstate(invalid="ignore", divide="ignore"):
        p = np.where(n > 0, k / n, np.nan)
        denom = 1 + z ** 2 / n
        centre = (p + z ** 2 / (2 * n)) / denom
        half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return centre - half, centre + half


def ci_band(ax, x, lo, hi, color, *, alpha=0.13):
    """Uncertainty band: no edge, low alpha, drawn under the line."""
    return ax.fill_between(x, lo, hi, color=color, alpha=alpha,
                           linewidth=0, zorder=1)


def layer_accuracy(df, variant, lens, *, prompt_type="train", prompt_idxs=None,
                   group_col=None):
    """Per-layer top-1 accuracy with a Wilson 95% CI, in percent.

    Mirrors viz._layer_top1_pooled but also returns the denominator, which the
    interval needs and which that helper discards.
    """
    sub = df[(df["variant"] == variant) & (df["lens"] == lens) &
             (df["prompt_type"] == prompt_type)]
    if prompt_idxs is not None:
        sub = sub[sub["prompt_idx"].isin(prompt_idxs)]
    if sub.empty:
        return pd.DataFrame(columns=["layer", "pct", "lo", "hi", "n"])
    keys = ["layer"] if group_col is None else [group_col, "layer"]
    out = (sub.groupby(keys, as_index=False)
           .agg(k=("in_top_1", "sum"), n=("in_top_1", "size")))
    lo, hi = wilson_ci(out["k"], out["n"])
    out["pct"] = out["k"] / out["n"] * 100.0
    out["lo"], out["hi"] = lo * 100.0, hi * 100.0
    return out


## Population reference

Every cell states its population explicitly, but the recurring vocabulary is:

- **variant** — `base` (pre-LoRA), `step_NNN` (intermediate checkpoint), `final`
  (last LoRA checkpoint).
- **condition** — `known` / `latent` (top-5 pre-LoRA) / `unknown`: real
  CounterFact facts stratified by whether the base model already knew them;
  `synthetic`: fabricated pseudo-entity facts the base model cannot have seen.
  `existing` = known ∪ latent ∪ unknown.
- **prompt_type** — `train` (facts LoRA was fine-tuned on) vs `paraphrase`
  (held-out generalization probes, never trained on).
- **lens** — `logit` (raw unembedding at each layer) vs `tuned` (a lens
  trained on the *base* model's activations; see README's lens-validity note).
- **first layer** — earliest layer at which the answer is top-1.
- **consistent layer** — earliest layer from which the answer is top-1
  continuously through to the output (`settle_layer` in the data).
- **H1** — does the correct answer emerge at an *earlier layer* after LoRA
  fine-tuning (shift in first / consistent layer)?
- **H2** — does that shift differ between facts the model already knew
  (existing conditions) and newly introduced facts (synthetic)?

Unless a figure says otherwise, it covers all four conditions together; any
split by condition or by existing/synthetic is stated in the caption.


## H1/H2 — does LoRA move the answer earlier, and does it differ by condition?


### Layer shift from base to LoRA

**Presents:** distribution of `base − LoRA` for the first layer and the
consistent layer, one histogram each; positive means LoRA made the answer
appear earlier. Both panels share a y-axis so the two are directly
comparable.

**Hypothesis:** H1 — does LoRA shift the answer to an earlier layer?

**Population:** train prompts, all four conditions, restricted to facts that
have a value for the given metric under *both* base and LoRA (a fact the base
model never gets right has no layer to shift from).


In [ ]:
lens = LENS
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory data available — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))]

    fig, axes = plt.subplots(1, 2, figsize=FIG_2COL, sharey=True)
    for ax, (col, label) in zip(axes, LAYER_METRICS):
        wide = sub.pivot_table(index="prompt_idx", columns="variant", values=col)
        if not {"base", "final"}.issubset(wide.columns):
            ax.set_visible(False)
            continue
        both = wide[["base", "final"]].dropna()
        if both.empty:
            title(ax, f"{label} (no data)")
            continue
        delta = both["base"] - both["final"]

        lo, hi = int(np.floor(delta.min())), int(np.ceil(delta.max()))
        ax.hist(delta, bins=np.arange(lo - 0.5, hi + 1.5, 1.0), color=C_LORA,
                alpha=0.75, edgecolor="white", linewidth=0.4)

        mean_d, med_d = float(delta.mean()), float(delta.median())
        pct_pos = 100.0 * (delta > 0).mean()
        ax.axvline(mean_d, color=C_ACCENT, linewidth=1.0, linestyle="--",
                   label=f"mean {mean_d:.1f}")
        ax.axvline(med_d, color=C_ACCENT, linewidth=1.0, linestyle=":",
                   label=f"median {med_d:.1f}")
        ax.set_xlabel(r"$\Delta$ layers (base $-$ LoRA); positive = earlier")
        title(ax, f"{label}   (n={len(delta)}, {pct_pos:.0f}% earlier)")
        ax.legend(loc="upper left")
        int_axis(ax)

    axes[0].set_ylabel("Prompts")
    suptitle(fig, "Emergence layer, base vs. LoRA", y=1.03)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "delta_layer_hist", lens=lens)


### Layer shift over the course of training

**Presents:** how the base→checkpoint layer shift develops during
fine-tuning. For every checkpoint the shift is computed per prompt against
that prompt's own base layer, then summarised as a mean and a median — the
extra checkpoint dimension makes per-prompt histograms unreadable, so only
the two central estimates are plotted. Positive means earlier than base.

**Hypothesis:** H1 over training time — is the earlier-layer effect present
from the first checkpoints, or does it accumulate gradually?

**Population:** train prompts, all four conditions, every checkpoint through
the final one; at each checkpoint, the facts with a value for the metric
under both base and that checkpoint.


In [ ]:
lens = LENS
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory data available — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train")]
    ckpts = (sub[sub["variant"] != "base"][["variant", "step"]]
             .drop_duplicates().sort_values("step"))
    if ckpts.empty:
        print("[notebook] no checkpoints beyond base — skipping.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=FIG_2COL, sharey=True)
        for ax, (col, label) in zip(axes, LAYER_METRICS):
            base_vals = sub[sub["variant"] == "base"].set_index("prompt_idx")[col].dropna()
            rows = []
            for variant, step in ckpts.itertuples(index=False):
                cur = sub[sub["variant"] == variant].set_index("prompt_idx")[col].dropna()
                b, c = base_vals.align(cur, join="inner")
                if b.empty:
                    continue
                delta = b - c
                rows.append({"step": step, "mean": delta.mean(),
                             "median": delta.median(), "n": len(delta),
                             "sem": delta.std(ddof=1) / np.sqrt(len(delta))})
            d = pd.DataFrame(rows)
            if d.empty:
                title(ax, f"{label} (no data)")
                continue

            ci_band(ax, d["step"], d["mean"] - 1.96 * d["sem"],
                    d["mean"] + 1.96 * d["sem"], C_LORA)
            ax.plot(d["step"], d["mean"], color=C_LORA, marker="o", label="mean")
            ax.plot(d["step"], d["median"], color=C_ACCENT, marker="s",
                    linestyle="--", label="median")
            ax.set_xlabel("Training step")
            title(ax, f"{label}   (n={int(d['n'].iloc[-1])} at final)")
            value_ticks(ax, d["step"])
            ax.tick_params(axis="x", labelsize=6, rotation=90)
            ax.legend(loc="best")

        axes[0].set_ylabel(r"$\Delta$ layers (base $-$ checkpoint)")
        suptitle(fig, "Layer shift over fine-tuning", y=1.03)
        fig.tight_layout(pad=0.4, w_pad=1.2)
        show_and_save(fig, "delta_layer_over_checkpoints", lens=lens)


### Distribution of emergence layers, base vs. LoRA

**Presents:** for each metric, how many prompts have their answer emerge at
each layer, drawn as a thin line per variant rather than as bars. Both panels
share x- and y-axes.

The legend gives, for each variant, the number of prompts the curve is built
from and what share of the whole prompt set that is. This matters: a prompt
whose answer is *never* top-1 has no emergence layer and contributes to no
bin, so the base curve sits lower largely because far fewer base facts are
ever correct — which is a result in itself, not a plotting artefact.

**Hypothesis:** H1, as the raw counterpart to the paired shift above — are
the two distributions genuinely displaced, or just differently shaped?

**Population:** train prompts, all four conditions; percentages are relative
to every train prompt in the run, including never-correct ones.


In [ ]:
lens = LENS
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory data available — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))]
    n_layers = int(np.nanmax([sub[c].max() for c, _ in LAYER_METRICS]))
    layers = np.arange(0, n_layers + 1)

    counts = {}
    for col, _ in LAYER_METRICS:
        for variant in ("base", "final"):
            v = sub[sub["variant"] == variant]
            vals = v[col].dropna()
            counts[(col, variant)] = (
                np.bincount(vals.astype(int), minlength=n_layers + 1)[:n_layers + 1],
                len(vals), v["prompt_idx"].nunique())
    ymax = max(c[0].max() for c in counts.values())

    fig, axes = plt.subplots(1, 2, figsize=FIG_2COL, sharex=True, sharey=True)
    for ax, (col, label) in zip(axes, LAYER_METRICS):
        for variant, color, name in (("base", C_BASE, "Base"),
                                     ("final", C_LORA, "LoRA")):
            y, n_have, n_all = counts[(col, variant)]
            pct = 100.0 * n_have / n_all if n_all else 0.0
            ax.plot(layers, y, color=color,
                    label=f"{name}  n={n_have} ({pct:.0f}% of {n_all})")
        title(ax, label)
        ax.legend(loc="upper left")
        layer_axis(ax, n_layers)

    axes[0].set_ylabel("Prompts")
    axes[0].set_ylim(0, ymax * 1.38)
    suptitle(fig, "Emergence layer distribution", y=1.03)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "layer_hist", lens=lens)


### Log-probability gained per layer

**Presents:** the mean change in the answer's log-probability at each layer,
LoRA minus base, with the median activation-patching first-flip layer marked
for reference.

**Hypothesis:** H1 — if LoRA relocates the answer earlier, the gain should
concentrate around the causal locus that patching identifies rather than
appearing only in the final layers.

**Population:** train prompts, all four conditions.


In [ ]:
lens = LENS
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df = df[(df["lens"] == lens) & (df["variant"].isin(["base", "final"])) &
            (df["prompt_type"] == "train")]

    mean_lp = df.groupby(["variant", "layer"])["answer_logprob"].mean().unstack(0)
    delta = (mean_lp["final"] - mean_lp["base"]).reset_index(name="delta_lp")
    n_layers = int(delta["layer"].max())

    fig, ax = plt.subplots(figsize=FIG_1COL)
    ax.plot(delta["layer"], delta["delta_lp"], color=C_LORA, marker="o",
            markevery=4, label=r"$\Delta$ log-probability")

    med = viz._pooled_median_first_flip(RESULTS_DIR)
    if med is not None:
        ax.axvline(med, color=C_ACCENT, linewidth=1.0, linestyle=":",
                   label=f"median first flip ({med:.0f})")

    ax.set_ylabel("Mean $\\Delta$ log-probability\n(LoRA $-$ base)")
    title(ax, "Log-probability gain")
    layer_axis(ax, n_layers)
    ax.legend(loc="upper left")

    fig.tight_layout(pad=0.5)
    show_and_save(fig, "delta_logprob", lens=lens)


### Which layer the LoRA update acts on

**Presents:** the distribution of the first-flip layer — the earliest layer
whose patched activation flips the base model to the LoRA answer — as one box
per condition, with every individual fact drawn as a point behind the box.
The count under each label is how many facts in that condition ever flip, out
of how many were patched.

**Hypothesis:** H2 — patching is lens-free, so this is the causal check on
whether latent / unknown / synthetic facts are written to different depths.

**Population:** final LoRA checkpoint, split by condition, lens-free
(patching operates on hidden states directly). Boxes cover only the facts
that flip; the ones that never flip are reported in the counts. Note that
`known` facts are not patched by the pipeline — the base model already
answers them correctly, so there is no flip to induce — and so the condition
is absent here by construction rather than for want of data.


In [ ]:
path = RESULTS_DIR / "patching.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    final = df[(df["layer"] == -1) & (df["variant"] == "final")]
    present = [c for c in COND_ORDER if c in final["condition"].values]
    n_layers = int(df.loc[df["layer"] >= 0, "layer"].max())

    fig, ax = plt.subplots(figsize=FIG_1COL_SQ)
    data, labels = [], []
    for cond in present:
        vals = final.loc[final["condition"] == cond, "first_flip_layer"].dropna()
        n_total = int((final["condition"] == cond).sum())
        data.append(vals)
        # Counts live in the tick label, so they can never collide with the title.
        labels.append(f"{CONDITION_LABELS[cond]}\n{len(vals)}/{n_total}")

    # Every patched fact as a point, jittered horizontally so ties separate;
    # the box sits on top as the summary rather than replacing the data.
    rng = np.random.default_rng(42)
    for i, vals in enumerate(data, start=1):
        ax.scatter(i + rng.uniform(-0.16, 0.16, len(vals)), vals,
                   color=COLORS[present[i - 1]], s=5, alpha=0.45,
                   linewidths=0, zorder=1)

    bp = ax.boxplot(data, widths=0.5, patch_artist=True, tick_labels=labels,
                    zorder=3,
                    medianprops=dict(color=C_NEUTRAL, linewidth=1.0),
                    boxprops=dict(linewidth=0.6, edgecolor=C_NEUTRAL),
                    whiskerprops=dict(linewidth=0.6, color=C_NEUTRAL),
                    capprops=dict(linewidth=0.6, color=C_NEUTRAL),
                    showfliers=False)
    for patch, cond in zip(bp["boxes"], present):
        patch.set_facecolor(COLORS[cond])
        patch.set_alpha(0.30)

    ax.set_ylim(-0.5, n_layers + 0.5)
    int_axis(ax, "y", step=4)
    ax.set_ylabel("First-flip layer")
    title(ax, "Patching: first-flip layer")
    ax.grid(axis="x", visible=False)

    fig.tight_layout(pad=0.5)
    show_and_save(fig, "patching")


### Depth of the causal effect

**Presents:** (a) the first-flip layer, and (b) the persistent-flip layer —
the earliest layer from which the patched flip holds all the way to the
output.

**Hypothesis:** H1's causal counterpart — is the flip a transient
perturbation or a durable one, and at what depth does it become durable?

**Population:** final LoRA checkpoint, all conditions, lens-free.


In [ ]:
path = RESULTS_DIR / "patching.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    first = df.loc[(df["layer"] == -1) & (df["variant"] == "final"),
                   "first_flip_layer"].dropna()
    persist = viz._persistent_flip_layer(df[df["variant"] == "final"])
    persist_vals = persist["persistent_flip_layer"].dropna()
    n_layers = int(df.loc[df["layer"] >= 0, "layer"].max())
    bins = np.arange(-0.5, n_layers + 1.5, 1.0)

    fig, axes = plt.subplots(1, 2, figsize=FIG_2COL, sharex=True, sharey=True)
    for ax, vals, color, label in (
            (axes[0], first, C_LORA, "First flip"),
            (axes[1], persist_vals, C_ACCENT, "Persistent flip")):
        ax.hist(vals, bins=bins, color=color, alpha=0.75,
                edgecolor="white", linewidth=0.4)
        title(ax, f"{label}   (n={len(vals)})")
        ax.set_xlabel("Layer")
        int_axis(ax)

    axes[0].set_ylabel("Facts")
    suptitle(fig, "Patching depth", y=1.03)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "patching_layer_hist")


### Depth of the causal effect, by condition

**Presents:** the same two depth distributions as above, split by condition
and drawn *cumulatively* — each curve is the share of that condition's
flipping facts that have flipped by the given layer. Per-layer shares are
too noisy to compare at these sample sizes (50–100 facts spread over 25
layers); the cumulative form removes that jitter and makes the ordering and
the medians readable directly off the 50% line. Normalising within condition
keeps groups with different flip counts comparable.

**Hypothesis:** H2 — do the conditions differ in *where* the causal effect
sits, not just in how often it fires?

**Population:** final LoRA checkpoint, split by condition, lens-free; only
facts that flip contribute. `known` is absent for the reason given above.


In [ ]:
path = RESULTS_DIR / "patching.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    final_sum = df[(df["layer"] == -1) & (df["variant"] == "final")]
    persist = viz._persistent_flip_layer(df[df["variant"] == "final"])
    n_layers = int(df.loc[df["layer"] >= 0, "layer"].max())
    layers = np.arange(0, n_layers + 1)
    present = [c for c in COND_ORDER if c in final_sum["condition"].values]

    panels = [("First flip", final_sum, "first_flip_layer"),
              ("Persistent flip", persist, "persistent_flip_layer")]

    fig, axes = plt.subplots(1, 2, figsize=FIG_2COL, sharex=True, sharey=True)
    for ax, (label, frame, col) in zip(axes, panels):
        for cond in present:
            vals = frame.loc[frame["condition"] == cond, col].dropna()
            if vals.empty:
                continue
            share = np.cumsum(
                np.bincount(vals.astype(int), minlength=n_layers + 1)[:n_layers + 1]
            ) / len(vals) * 100.0
            ax.plot(layers, share, color=COLORS[cond],
                    linestyle=LINE_STYLES[cond],
                    label=f"{CONDITION_LABELS[cond]} (n={len(vals)})")
        ax.axhline(50, color=C_NEUTRAL, linewidth=0.6, linestyle=":")
        title(ax, label)
        ax.set_xlabel("Layer")
        ax.set_ylim(0, 102)
        int_axis(ax)

    axes[0].set_ylabel("Cumulative % of\nflipping facts")
    axes[0].legend(loc="upper left")
    suptitle(fig, "Patching depth by condition", y=1.03)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "patching_layer_hist_by_condition")


## Summary tables


### Table 1 — accuracy and log-probability, before vs. after LoRA

**Presents:** Acc@1 and mean answer log-probability at the output, base vs.
LoRA, for training prompts and for held-out paraphrases. Values are
`n_prompts`-weighted averages across conditions.

Both quantities are read at the final layer, which is the model's actual
output distribution — so they are identical under either lens (verified: the
logit- and tuned-lens columns agree exactly). The table is therefore reported
once, without a lens split.

**Hypothesis:** H1 headline numbers — the scalar summary behind the
layer-shift figures above.

**Population:** train and paraphrase prompts, all four conditions.


In [ ]:
summary_path = RESULTS_DIR / "summary.csv"
if not summary_path.exists():
    print(f"[notebook] {summary_path} not found — skipping.")
else:
    s_all = pd.read_csv(summary_path)
    # Acc@1 / log-prob are final-layer quantities and so are lens-invariant;
    # assert that here rather than silently picking one lens.
    chk = s_all.pivot_table(index=["variant", "condition", "prompt_type"],
                            columns="lens",
                            values=["final_accuracy", "mean_final_logprob"])
    if "tuned" in s_all["lens"].unique():
        for metric in ("final_accuracy", "mean_final_logprob"):
            gap = (chk[(metric, "logit")] - chk[(metric, "tuned")]).abs().max()
            assert gap < 1e-9, f"{metric} differs between lenses by {gap}"

    s = s_all[(s_all["lens"] == "logit") & (s_all["variant"].isin(["base", "final"]))]

    def _wavg(frame, col):
        if "n_prompts" in frame.columns and frame["n_prompts"].sum():
            return np.average(frame[col], weights=frame["n_prompts"])
        return frame[col].mean()

    rows = []
    for ptype, label in (("train", "train"), ("paraphrase", "paraphrase")):
        b = s[(s["variant"] == "base") & (s["prompt_type"] == ptype)]
        f = s[(s["variant"] == "final") & (s["prompt_type"] == ptype)]
        if b.empty or f.empty:
            continue
        rows.append({
            "prompts":      label,
            "acc@1_base":   _wavg(b, "final_accuracy"),
            "acc@1_lora":   _wavg(f, "final_accuracy"),
            "logprob_base": _wavg(b, "mean_final_logprob"),
            "logprob_lora": _wavg(f, "mean_final_logprob"),
        })
    table1 = pd.DataFrame(rows).set_index("prompts").round(3)
    display(table1)


### Table 1b — logit-lens vs. tuned-lens agreement

**Presents:** Acc@1 and mean first layer per condition under each lens, at
the final checkpoint. Acc@1 is a final-layer quantity and so agrees exactly
by construction; the first-layer columns are the informative comparison.

**Hypothesis:** the lens-validity check the README calls out — the tuned lens
is fit on the *base* model, so the two lenses agreeing on depth is what
licenses trusting either one's H1/H2 conclusions.

**Population:** train prompts, final checkpoint, split by condition, both
lenses required.


In [ ]:
summary_path = RESULTS_DIR / "summary.csv"
if not summary_path.exists():
    print(f"[notebook] {summary_path} not found — skipping.")
else:
    s_all = pd.read_csv(summary_path)
    cmp_df = s_all[(s_all["variant"] == "final") & (s_all["prompt_type"] == "train")]
    if "tuned" not in cmp_df["lens"].unique():
        print("[notebook] no tuned-lens rows — skipping.")
    else:
        rows = []
        for cond in COND_ORDER:
            lg = cmp_df[(cmp_df["condition"] == cond) & (cmp_df["lens"] == "logit")]
            tn = cmp_df[(cmp_df["condition"] == cond) & (cmp_df["lens"] == "tuned")]
            if lg.empty or tn.empty:
                continue
            lg, tn = lg.iloc[0], tn.iloc[0]
            rows.append({
                "condition":         CONDITION_LABELS[cond],
                "acc@1_logit":       lg["final_accuracy"],
                "acc@1_tuned":       tn["final_accuracy"],
                "first_layer_logit": lg["mean_first_layer"],
                "first_layer_tuned": tn["mean_first_layer"],
            })
        table1b = pd.DataFrame(rows).set_index("condition").round(3)
        display(table1b)


### Table 1c — trajectory class shares

**Presents:** the share of facts in each trajectory class — `never`,
`transient`, `late_only`, `persistent` — before and after LoRA.

**Hypothesis:** H1 — a persistent share that grows at the expense of `never`
and `transient` is the trajectory-level signature of the answer emerging and
*staying* correct.

**Population:** train prompts, all four conditions. Requires the `trajectory`
pipeline stage to have written `trajectory_summary.csv`.


In [ ]:
lens = LENS
path = RESULTS_DIR / "trajectory_summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found for this run — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train") &
          (s["variant"].isin(["base", "final"]))]
    if s.empty:
        print("[notebook] no rows for this lens — skipping.")
    else:
        rows = []
        for variant, label in (("base", "base"), ("final", "LoRA")):
            sub = s[s["variant"] == variant]
            totals = {c: sub[c].sum() if c in sub.columns else 0 for c in CLASS_ORDER}
            n = sum(totals.values()) or 1
            rows.append({"variant": label, **{c: totals[c] / n for c in CLASS_ORDER}})
        display(pd.DataFrame(rows).set_index("variant").round(3))


## Layer dynamics, generalization, and rank


### Layer-wise accuracy on facts the base model could already surface

**Presents:** the share of facts whose answer is top-1 at each layer, base vs.
LoRA, restricted to facts the base model surfaced at *some* layer before
fine-tuning.

**Hypothesis:** H1 on the cleanest subset — for facts the base model already
holds somewhere in its forward pass, does LoRA pull the answer earlier, or
merely sharpen it at the same depth?

**Population:** train prompts; facts whose base first layer is non-null at
any layer ("lens-detectable"), all four conditions.


In [ ]:
lens = LENS
path = RESULTS_DIR / "layerwise.parquet"
traj = viz._load_traj(RESULTS_DIR)
if not path.exists() or traj is None:
    print("[notebook] missing data — skipping.")
else:
    idxs = viz.detectable_prompt_idxs(traj, lens)
    if not idxs:
        print("[notebook] no detectable prompts — skipping.")
    else:
        df = pd.read_parquet(path)
        base = layer_accuracy(df, "base", lens, prompt_idxs=idxs)
        final = layer_accuracy(df, "final", lens, prompt_idxs=idxs)
        n_layers = int(max(base["layer"].max(), final["layer"].max()))

        fig, ax = plt.subplots(figsize=FIG_1COL)
        for d, color, ls, name in ((base, C_BASE, "--", "Base"),
                                   (final, C_LORA, "-", "LoRA")):
            ci_band(ax, d["layer"], d["lo"], d["hi"], color)
            ax.plot(d["layer"], d["pct"], color=color, linestyle=ls, label=name)
        ax.set_ylabel("Top-1 accuracy (%)")
        title(ax, f"Lens-detectable facts (n={len(idxs)})")
        layer_axis(ax, n_layers)
        ax.legend(loc="upper left")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "layer_top1_detectable", lens=lens)


### Layer-wise accuracy on all facts

**Presents:** the same curve as above without the detectability restriction —
every training fact, base vs. LoRA.

**Hypothesis:** H1 at the full-population level, the counterpart to the
detectable-only subset above.

**Population:** train prompts, all four conditions.


In [ ]:
lens = LENS
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    base = layer_accuracy(df, "base", lens)
    final = layer_accuracy(df, "final", lens)
    n_layers = int(max(base["layer"].max(), final["layer"].max()))

    fig, ax = plt.subplots(figsize=FIG_1COL)
    for d, color, ls, name in ((base, C_BASE, "--", "Base"),
                               (final, C_LORA, "-", "LoRA")):
        ci_band(ax, d["layer"], d["lo"], d["hi"], color)
        ax.plot(d["layer"], d["pct"], color=color, linestyle=ls, label=name)
    ax.set_ylabel("Top-1 accuracy (%)")
    title(ax, "All facts")
    layer_axis(ax, n_layers)
    ax.legend(loc="upper left")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_top1_all_facts", lens=lens)


### Layer-wise accuracy on repaired paraphrases

**Presents:** post-LoRA layer-wise accuracy on the paraphrase prompts of
facts where LoRA specifically closed the generalization gap — correct on the
training prompt both before and after, wrong on at least one paraphrase
before, correct on the paraphrase after. The original training prompt for the
same facts is drawn alongside, so the paraphrase curve can be read against
the phrasing the model was actually trained on.

**Hypothesis:** H1 on the subset where LoRA taught genuine generalization —
at what layer does the now-correct paraphrase answer emerge, and does it lag
the trained phrasing?

**Population:** final checkpoint; facts matching the repair filter above,
with their paraphrase and training prompts plotted separately. Typically a
small n.


In [ ]:
lens = LENS
traj = viz._load_traj(RESULTS_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if traj is None or not path.exists():
    print("[notebook] missing data — skipping.")
else:
    t = traj[(traj["lens"] == lens) & (traj["variant"].isin(["base", "final"]))]
    train = t[t["prompt_type"] == "train"]
    para = t[t["prompt_type"] == "paraphrase"]
    if train.empty or para.empty:
        print("[notebook] no paraphrase rows — skipping.")
    else:
        tr = train.set_index(["fact_id", "variant"])
        pr = para.set_index(["fact_id", "variant"])
        survives = "traj_class in ['late_only','persistent']"
        base_train_ok = set(tr.loc[(slice(None), "base"), :]
                            .query(survives).index.get_level_values(0))
        final_train_ok = set(tr.loc[(slice(None), "final"), :]
                             .query(survives).index.get_level_values(0))
        base_para = pr.loc[(slice(None), "base"), :]
        base_para_bad = set(base_para.loc[
            ~base_para["traj_class"].isin(["late_only", "persistent"])
        ].index.get_level_values(0))
        final_para_ok = set(pr.loc[(slice(None), "final"), :]
                            .query(survives).index.get_level_values(0))

        facts = base_train_ok & base_para_bad & final_para_ok & final_train_ok
        if not facts:
            print("[notebook] filter matched 0 facts — skipping.")
        else:
            df = pd.read_parquet(path)
            para_idxs = set(para.loc[(para["variant"] == "final") &
                                     para["fact_id"].isin(facts), "prompt_idx"])
            train_idxs = set(train.loc[(train["variant"] == "final") &
                                       train["fact_id"].isin(facts), "prompt_idx"])
            curve_para = layer_accuracy(df, "final", lens,
                                        prompt_type="paraphrase",
                                        prompt_idxs=para_idxs)
            curve_train = layer_accuracy(df, "final", lens,
                                         prompt_type="train",
                                         prompt_idxs=train_idxs)
            n_layers = int(curve_para["layer"].max())

            fig, ax = plt.subplots(figsize=FIG_1COL)
            for d, color, ls, name in ((curve_train, C_BASE, "--", "Original prompt"),
                                       (curve_para, C_LORA, "-", "Paraphrase")):
                ci_band(ax, d["layer"], d["lo"], d["hi"], color)
                ax.plot(d["layer"], d["pct"], color=color, linestyle=ls, label=name)
            ax.set_ylabel("Top-1 accuracy (%)")
            title(ax, f"Repaired paraphrases (n={len(facts)} facts)")
            layer_axis(ax, n_layers)
            ax.legend(loc="upper left")
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "paraphrase_layers_after_lora", lens=lens)


### Extra depth required by a paraphrase

**Presents:** the distribution of `L_para − L_orig`, the difference between
the layer at which the answer first appears on a paraphrase and on the
original training prompt, after LoRA. Positive means the paraphrase needs
more depth. Facts never correct on one side or the other have no difference
to report and are counted separately in the title.

**Hypothesis:** H1's generalization-depth counterpart — even where LoRA does
generalize, does the reworded prompt take extra layers to get there?

**Population:** final checkpoint, all four conditions; facts with a first
layer on both the training prompt and (the earliest of) its paraphrases.


In [ ]:
lens = LENS
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["variant"] == "final")]
    train = (sub[sub["prompt_type"] == "train"][["fact_id", "condition", "first_layer"]]
             .rename(columns={"first_layer": "L_orig"}))
    para = (sub[sub["prompt_type"] == "paraphrase"]
            .groupby(["fact_id", "condition"], as_index=False)
            .agg(L_para=("first_layer", "min")))
    joined = train.merge(para, on=["fact_id", "condition"], how="inner")
    never = joined[joined["L_orig"].isna() | joined["L_para"].isna()]
    both = joined.dropna(subset=["L_orig", "L_para"]).copy()
    both["delta"] = both["L_para"] - both["L_orig"]

    fig, ax = plt.subplots(figsize=FIG_1COL)
    if not both.empty:
        lo, hi = int(np.floor(both["delta"].min())), int(np.ceil(both["delta"].max()))
        ax.hist(both["delta"], bins=np.arange(lo - 0.5, hi + 1.5, 1.0),
                color=C_LORA, alpha=0.75, edgecolor="white", linewidth=0.4)
        ax.axvline(both["delta"].mean(), color=C_ACCENT, linestyle="--",
                   linewidth=1.0, label=f"mean {both['delta'].mean():.1f}")
        ax.legend(loc="upper left")
    ax.set_xlabel(r"$L_{\mathrm{para}} - L_{\mathrm{orig}}$ (layers)")
    ax.set_ylabel("Facts")
    int_axis(ax)
    title(ax, f"Paraphrase depth penalty (n={len(both)}, {len(never)} excluded)")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "paraphrase_layer_delay", lens=lens)


### Fact-state transitions

The next two figures use a three-state collapse of the trajectory classes:
`never`, `transient`, and `survives` (`late_only` ∪ `persistent`). The helper
below draws the alluvial diagram; labels sit outside the node bars so they
cannot overflow them, and ribbons are drawn largest-first with a soft
S-curve.


In [ ]:
SANKEY_STATES = ["never", "transient", "survives"]
SANKEY_COLORS = {"never": OKABE_ITO["grey"],
                 "transient": OKABE_ITO["orange"],
                 "survives": OKABE_ITO["blue"]}


def draw_sankey(ax, flows, *, left_label="Base", right_label="LoRA",
                gap=0.05, fontsize=5.5):
    """Three-state alluvial diagram between base (left) and LoRA (right).

    `flows` maps (base_state, final_state) -> count. Node labels are placed
    outside the bars, so long state names and counts never have to fit inside
    a box. Both columns are laid out against the same available height, so a
    ribbon's thickness matches its endpoints on either side.
    """
    states = SANKEY_STATES
    left_tot = {s: sum(n for (a, _), n in flows.items() if a == s) for s in states}
    right_tot = {s: sum(n for (_, b), n in flows.items() if b == s) for s in states}
    total = max(sum(left_tot.values()), 1)
    avail = 1.0 - gap * (len(states) - 1)

    def spans(tots):
        y, out = 1.0, {}
        for s in states:
            h = avail * tots[s] / total
            out[s] = (y - h, y) if tots[s] else None
            y -= h + gap
        return out

    left, right = spans(left_tot), spans(right_tot)
    x0, x1, bw = 0.0, 1.0, 0.035
    lc = {s: left[s][1] for s in states if left[s]}
    rc = {s: right[s][1] for s in states if right[s]}

    t = np.linspace(0.0, 1.0, 120)
    smooth = t * t * (3.0 - 2.0 * t)          # smoothstep S-curve
    xs = (x0 + bw) + ((x1 - bw) - (x0 + bw)) * t

    for (a, b), n in sorted(flows.items(), key=lambda kv: -kv[1]):
        if n <= 0 or left[a] is None or right[b] is None:
            continue
        h = avail * n / total
        top_l, top_r = lc[a], rc[b]
        bot_l, bot_r = top_l - h, top_r - h
        lc[a], rc[b] = bot_l, bot_r
        ax.fill_between(xs,
                        bot_l + (bot_r - bot_l) * smooth,
                        top_l + (top_r - top_l) * smooth,
                        color=SANKEY_COLORS[a], alpha=0.22, linewidth=0)

    for s in states:
        if left[s]:
            b0, t0 = left[s]
            ax.add_patch(plt.Rectangle((x0, b0), bw, t0 - b0,
                                       facecolor=SANKEY_COLORS[s], edgecolor="none"))
            ax.text(x0 - 0.03, (b0 + t0) / 2, f"{s}  {left_tot[s]}",
                    ha="right", va="center", fontsize=fontsize)
        if right[s]:
            b1, t1 = right[s]
            ax.add_patch(plt.Rectangle((x1 - bw, b1), bw, t1 - b1,
                                       facecolor=SANKEY_COLORS[s], edgecolor="none"))
            ax.text(x1 + 0.03, (b1 + t1) / 2, f"{right_tot[s]}  {s}",
                    ha="left", va="center", fontsize=fontsize)

    ax.text(x0 + bw / 2, 1.05, left_label, ha="center", va="bottom", fontsize=fontsize)
    ax.text(x1 - bw / 2, 1.05, right_label, ha="center", va="bottom", fontsize=fontsize)
    ax.set_xlim(-0.46, 1.46)
    ax.set_ylim(-0.03, 1.14)
    ax.axis("off")


def state_flows(frame):
    wide = frame.pivot_table(index="prompt_idx", columns="variant",
                             values="state", aggfunc="first").dropna()
    if wide.empty or not {"base", "final"}.issubset(wide.columns):
        return {}
    return {(a, b): int(n) for (a, b), n in wide.groupby(["base", "final"]).size().items()}


### Fact-state transitions, base to LoRA

**Presents:** how facts move between the three states from base to LoRA;
ribbon thickness is the number of facts making that transition.

**Hypothesis:** H1 as a state-transition story — mass flowing out of `never`
and `transient` into `survives` is the qualitative signature of LoRA making
facts stick.

**Population:** train prompts, all four conditions.


In [ ]:
lens = LENS
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))].copy()
    sub["state"] = viz.three_state(sub["traj_class"])

    fig, ax = plt.subplots(figsize=FIG_1COL_SQ)
    draw_sankey(ax, state_flows(sub))
    title(ax, "Fact-state transitions", pad=14)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "fact_state_sankey", lens=lens)


### Fact-state transitions by condition

**Presents:** the same alluvial diagram drawn once per condition.

**Hypothesis:** H2 — do known / latent / unknown / synthetic facts move
between states in different proportions?

**Population:** train prompts, split by condition.


In [ ]:
lens = LENS
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))].copy()
    sub["state"] = viz.three_state(sub["traj_class"])
    present = [c for c in COND_ORDER if c in sub["condition"].values]

    if not present:
        print("[notebook] no conditions present — skipping.")
    else:
        fig, axes = plt.subplots(1, len(present), figsize=(1.75 * len(present), 2.2))
        axes = np.atleast_1d(axes)
        for ax, cond in zip(axes, present):
            draw_sankey(ax, state_flows(sub[sub["condition"] == cond]), fontsize=5)
            title(ax, CONDITION_LABELS[cond], pad=12)
        suptitle(fig, "Fact-state transitions by condition", y=1.04)
        fig.tight_layout(pad=0.4, w_pad=1.6)
        show_and_save(fig, "fact_state_sankey_by_condition", lens=lens)


### Accuracy against LoRA rank

**Presents:** final Acc@1 as a function of the LoRA rank $r$, one line per
condition, from the rank-ablation sweep ($r=0$ is the base model).

**Hypothesis:** H2's capacity angle — does adapter rank matter differently
for facts the model already knew than for fabricated ones?

**Population:** train prompts, split by condition, one re-trained adapter per
swept rank.


In [ ]:
lens = LENS
path = RESULTS_DIR / "rank_ablation_summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train")]

    fig, ax = plt.subplots(figsize=FIG_1COL)
    for cond in COND_ORDER:
        sub = df[df["condition"] == cond].sort_values("rank")
        if sub.empty:
            continue
        ax.plot(sub["rank"], sub["final_accuracy"], color=COLORS[cond],
                linestyle=LINE_STYLES[cond], marker=MARKERS[cond],
                label=CONDITION_LABELS[cond])
    ax.set_xlabel("LoRA rank $r$   ($r=0$: base model)")
    value_ticks(ax, df["rank"])
    ax.set_ylabel("Acc@1")
    title(ax, "Accuracy vs. rank")
    ax.set_ylim(-0.03, 1.03)
    ax.legend(loc="lower right")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "rank_final_accuracy", lens=lens)


### Emergence depth against LoRA rank

**Presents:** mean first layer and mean consistent layer as a function of
adapter rank, one line per condition. The consistent layer is not in
`rank_ablation_summary.csv`, so both are recomputed from
`rank_ablation_layerwise.parquet` with the same trajectory routine the
pipeline uses.

**Hypothesis:** H1 crossed with capacity — does a larger adapter move the
answer earlier, or only make it more often correct at the same depth (which
would show as a flat line here against a rising line in the accuracy figure
above)?

**Population:** train prompts, split by condition, one re-trained adapter per
swept rank; facts with a value for the metric at that rank. The base model
($r=0$) is excluded: by the definition of latent / unknown / synthetic its
answer is not top-1 at the output, so three of the four conditions have no
consistent layer there at all and the point would not be comparable.


In [ ]:
lens = LENS
path = RESULTS_DIR / "rank_ablation_layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    ral = pd.read_parquet(path)
    ral = ral[(ral["lens"] == lens) & (ral["prompt_type"] == "train")]
    rank_of = (ral[["variant", "rank"]].drop_duplicates()
               .set_index("variant")["rank"].to_dict())

    rtraj = per_prompt_trajectory(ral)
    rtraj["rank"] = rtraj["variant"].map(rank_of)

    # Drop the base model: three of the four conditions have no consistent
    # layer at r=0 by construction, so the point is not comparable.
    rtraj = rtraj[rtraj["rank"] > 0]

    agg = (rtraj.groupby(["condition", "rank"])
           .agg(first_layer=("first_layer", "mean"),
                settle_layer=("settle_layer", "mean"))
           .reset_index())

    fig, axes = plt.subplots(1, 2, figsize=FIG_2COL, sharex=True, sharey=True)
    for ax, (col, label) in zip(axes, LAYER_METRICS):
        for cond in COND_ORDER:
            sub = agg[agg["condition"] == cond].sort_values("rank")
            if sub.empty:
                continue
            ax.plot(sub["rank"], sub[col], color=COLORS[cond],
                    linestyle=LINE_STYLES[cond], marker=MARKERS[cond],
                    label=CONDITION_LABELS[cond])
        ax.set_xlabel("LoRA rank $r$")
        title(ax, f"Mean {label.lower()}")
        value_ticks(ax, agg["rank"])

    axes[0].set_ylabel("Layer")
    axes[0].legend(loc="best")
    suptitle(fig, "Emergence depth vs. rank", y=1.03)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "rank_layer_depth", lens=lens)


## Existing vs. fabricated facts

`existing` = known ∪ latent ∪ unknown (real CounterFact facts); `synthetic` =
fabricated pseudo-entity facts built from the same relation templates, which
the base model cannot have seen anywhere. This is H2's control: any
generalization the model shows on synthetic facts has to come from LoRA
training rather than prior knowledge.


### Layer-wise accuracy, existing vs. fabricated facts

**Presents:** the share of facts with the answer top-1 at each layer, base
(dashed) and LoRA (solid), for existing and for synthetic facts.

**Hypothesis:** H2 control — synthetic facts should carry no base-model
signal at all, so their entire curve has to be built by fine-tuning.

**Population:** train prompts, split into existing vs. synthetic.


In [ ]:
lens = LENS
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train")]
    df["group"] = viz.existing_vs_synthetic(df["condition"])
    n_layers = int(df["layer"].max())

    fig, ax = plt.subplots(figsize=FIG_1COL)
    for group, color in (("existing", COLORS["existing"]),
                         ("synthetic", COLORS["synthetic"])):
        for variant, ls, name in (("base", "--", "base"), ("final", "-", "LoRA")):
            sub = df[(df["group"] == group) & (df["variant"] == variant)]
            if sub.empty:
                continue
            agg = (sub.groupby("layer", as_index=False)
                   .agg(k=("in_top_1", "sum"), n=("in_top_1", "size")))
            lo, hi = wilson_ci(agg["k"], agg["n"])
            ci_band(ax, agg["layer"], lo * 100, hi * 100, color)
            ax.plot(agg["layer"], agg["k"] / agg["n"] * 100, color=color,
                    linestyle=ls, label=f"{group}, {name}")
    ax.set_ylabel("Top-1 accuracy (%)")
    title(ax, "Existing vs. fabricated facts")
    layer_axis(ax, n_layers)
    ax.legend(loc="upper left", ncol=2)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_top1_existing_vs_synthetic",
                  prefix="new_version_synthetic", lens=lens)


### Log-probability gained per layer, existing vs. fabricated facts

**Presents:** the mean LoRA-minus-base change in the answer's
log-probability at each layer, one curve per group.

**Hypothesis:** H1/H2 control — does LoRA add probability mass at the same
depths for facts it is memorising outright as for facts it already partly
knew?

**Population:** train prompts, split into existing vs. synthetic.


In [ ]:
lens = LENS
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train") &
            (df["variant"].isin(["base", "final"]))]
    df["group"] = viz.existing_vs_synthetic(df["condition"])
    n_layers = int(df["layer"].max())

    fig, ax = plt.subplots(figsize=FIG_1COL)
    for group, color in (("existing", COLORS["existing"]),
                         ("synthetic", COLORS["synthetic"])):
        mean_lp = (df[df["group"] == group]
                   .groupby(["variant", "layer"])["answer_logprob"].mean().unstack(0))
        if not {"base", "final"}.issubset(mean_lp.columns):
            continue
        ax.plot(mean_lp.index, mean_lp["final"] - mean_lp["base"],
                color=color, marker="o", markevery=4, label=group)
    ax.set_ylabel("Mean $\\Delta$ log-probability\n(LoRA $-$ base)")
    title(ax, "Log-probability gain")
    layer_axis(ax, n_layers)
    ax.legend(loc="upper left")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_delta_existing_vs_synthetic",
                  prefix="new_version_synthetic", lens=lens)


### Accuracy over training, existing vs. fabricated facts

**Presents:** Acc@1 at each checkpoint, one line per group.

**Hypothesis:** H2's learning-dynamics control — facts with no prior support
may need more optimisation to reach the same accuracy.

**Population:** train prompts, every checkpoint through the final one, split
into existing vs. synthetic.


In [ ]:
lens = LENS
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train")].copy()
    if not s["variant"].str.startswith("step_", na=False).any():
        print("[notebook] no intermediate checkpoints — skipping.")
    else:
        s["group"] = viz.existing_vs_synthetic(s["condition"])
        use = s[s["variant"].str.startswith("step_", na=False) | (s["variant"] == "final")]

        fig, ax = plt.subplots(figsize=FIG_1COL)
        for group, color in (("existing", COLORS["existing"]),
                             ("synthetic", COLORS["synthetic"])):
            g = (use[use["group"] == group].groupby("step")["final_accuracy"]
                 .mean().reset_index())
            ax.plot(g["step"], g["final_accuracy"], color=color, marker="o",
                    label=group)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Acc@1")
        title(ax, "Accuracy over training")
        ax.set_ylim(0, 1.05)
        ax.legend(loc="lower right")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "acc_over_checkpoints",
                      prefix="new_version_synthetic", lens=lens)


### Emergence depth over training, existing vs. fabricated facts

**Presents:** the mean first layer at each checkpoint, one line per group,
annotated at the final checkpoint with the share of facts that are never
top-1 (those contribute no first layer, so the mean is taken over the rest).

**Hypothesis:** H1's learning-dynamics control — does the depth for
fabricated facts converge toward that of real ones as training proceeds, or
stay persistently different?

**Population:** train prompts, every checkpoint through the final one, split
into existing vs. synthetic.


In [ ]:
lens = LENS
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train")].copy()
    if not s["variant"].str.startswith("step_", na=False).any():
        print("[notebook] no intermediate checkpoints — skipping.")
    else:
        s["group"] = viz.existing_vs_synthetic(s["condition"])
        use = s[s["variant"].str.startswith("step_", na=False) | (s["variant"] == "final")]

        fig, ax = plt.subplots(figsize=FIG_1COL)
        for group, color in (("existing", COLORS["existing"]),
                             ("synthetic", COLORS["synthetic"])):
            g = (use[use["group"] == group].groupby("step")
                 .agg(mean_first=("mean_first_layer", "mean"),
                      frac_never=("frac_never_top1", "mean"))
                 .reset_index())
            ax.plot(g["step"], g["mean_first"], color=color, marker="o", label=group)
            if not g.empty:
                last = g.iloc[-1]
                ax.annotate(f"never {last['frac_never']:.0%}",
                            (last["step"], last["mean_first"]),
                            textcoords="offset points", xytext=(5, 3),
                            fontsize=6.5, color=color)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Mean first layer")
        title(ax, "Emergence depth over training")
        ax.margins(x=0.14)
        ax.legend(loc="best")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "first_layer_over_checkpoints",
                      prefix="new_version_synthetic", lens=lens)


### Patching flip rate, existing vs. fabricated facts

**Presents:** the share of facts whose prediction flips when the activation
at each layer is patched, one curve per group.

**Hypothesis:** H2's causal, lens-free confirmation of the layer-wise
accuracy figure — does the causal locus sit at a comparable depth for
fabricated facts as for real ones?

**Population:** final LoRA checkpoint, all layers, split into existing vs.
synthetic, lens-free.


In [ ]:
path = RESULTS_DIR / "patching.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    df = df[(df["variant"] == "final") & (df["layer"] >= 0)].copy()
    df["group"] = viz.existing_vs_synthetic(df["condition"])
    n_layers = int(df["layer"].max())

    fig, ax = plt.subplots(figsize=FIG_1COL)
    for group, color in (("existing", COLORS["existing"]),
                         ("synthetic", COLORS["synthetic"])):
        rate = df[df["group"] == group].groupby("layer")["flipped"].mean()
        ax.plot(rate.index, rate.values * 100, color=color, marker="o",
                markevery=4, label=group)
    ax.set_ylabel("% of facts flipped by patching")
    title(ax, "Patching flip rate")
    layer_axis(ax, n_layers)
    ax.legend(loc="upper left")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "patching_existing_vs_synthetic",
                  prefix="new_version_synthetic")
